# Machine Learning – Practical Session 1
## Python for Data Exploration and Preprocessing

National Higher School of Advanced Technologies – Master 1 – 2026–2027

Fichiers utilisés : `Iris_Dataset.csv`, `txt_file.txt`, `employees.csv` (placés dans le dossier* `data`).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_20newsgroups
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler,
                                   normalize, Binarizer, OrdinalEncoder, LabelEncoder, OneHotEncoder)

def find_file(name):
    """Cherche un fichier dans le dossier courant puis dans quelques dossiers data."""
    for folder in [".", "data", "../data", "../../data"]:
        p = Path(folder) / name
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} introuvable : place-le à côté du notebook ou dans un dossier 'data'.")

# Exercice 1 — Exploration d'un jeu de données tabulaire et création des ensembles

### 1. Chargement du fichier CSV dans un DataFrame `D`

In [ ]:
D = pd.read_csv(find_file("Iris_Dataset.csv"))
D.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### 2. Inspection du jeu de données

In [ ]:
D.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


In [ ]:
D.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [ ]:
print("shape   :", D.shape)
print("columns :", list(D.columns))

shape   : (150, 5)
columns : ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


In [ ]:
D.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


**Ce que chaque sortie nous apprend :**

- `info()` : le nombre de lignes, le nom et le type (`dtype`) de chaque colonne, le nombre de valeurs non nulles (donc la présence de valeurs manquantes) et la mémoire utilisée.
- `describe()` : les statistiques descriptives des colonnes numériques (effectif, moyenne, écart-type, min, quartiles, max). Utile pour repérer les échelles et les valeurs aberrantes.
- `shape` : le couple (nombre d'observations, nombre de colonnes).
- `columns` : la liste des noms de colonnes (variables et éventuelle étiquette de classe).
- `head()` : un aperçu des premières lignes pour vérifier que le chargement est correct.

### 3. Sélection des deux variables `sepal_length` et `petal_width`

In [ ]:
D_two = D[["sepal_length", "petal_width"]]
D_two.shape

(150, 2)

### 4. Affichage des 20 premières observations

In [9]:
D_two.head(20)

,sepal_length,petal_width
0,5.1,0.2
1,4.9,0.2
2,4.7,0.2
3,4.6,0.2
4,5.0,0.2
5,5.4,0.4
6,4.6,0.3
7,5.0,0.2
8,4.4,0.2
9,4.9,0.1


### 5. Copie du DataFrame original

In [10]:
D_copy = D.copy()
print("Même objet ?", D_copy is D)

Même objet ? False


Une copie permet d'expérimenter (mise à l'échelle, suppression de colonnes, imputation...) **sans modifier les données originales**. Si une transformation échoue ou n'est pas satisfaisante, on repart de `D` intact au lieu de recharger le fichier. `.copy()` crée un objet indépendant, alors qu'une simple affectation `D2 = D` ne ferait que créer une seconde référence vers les mêmes données.

### 6-7. Séparation train / test (80 % / 20 %)

La variable cible est la dernière colonne (étiquette de classe) ; toutes les autres colonnes sont les variables d'entrée. On utilise `stratify=y` pour conserver les proportions de classes.

In [11]:
target_col = D.columns[-1]                # étiquette de classe (cible)
feature_cols = [c for c in D.columns if c != target_col]
X = D[feature_cols]
y = D[target_col]
print("Variables d'entrée :", feature_cols)
print("Variable cible     :", target_col)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Variables d'entrée : ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
Variable cible     : species


### 8. Vérification des effectifs et des proportions

In [12]:
print("Train :", X_train.shape, "->", round(len(X_train) / len(D) * 100, 1), "%")
print("Test  :", X_test.shape,  "->", round(len(X_test) / len(D) * 100, 1), "%")

print("\nProportion des classes :")
pd.DataFrame({
    "Dataset complet": y.value_counts(normalize=True),
    "Train": y_train.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True),
}).round(3)

Train : (120, 4) -> 80.0 %
Test  : (30, 4) -> 20.0 %

Proportion des classes :


,Dataset complet,Train,Test
species,,,
Iris-setosa,0.333,0.333,0.333
Iris-versicolor,0.333,0.333,0.333
Iris-virginica,0.333,0.333,0.333


**Discussion : pourquoi garder l'ensemble de test séparé jusqu'à l'évaluation finale ?**

Le test set simule des données **jamais vues**. Si de l'information du test influence le prétraitement (par exemple calculer la moyenne et l'écart-type sur tout le jeu de données avant de standardiser) ou le choix du modèle (régler les hyperparamètres en regardant le score de test), il y a **fuite de données (data leakage)**. Le score obtenu devient alors trop optimiste et ne reflète plus la vraie performance de généralisation. Pour régler les hyperparamètres, on utilise un ensemble de validation ou la validation croisée sur le train, et on ne touche au test qu'une seule fois, à la fin.

# Exercice 2 — Exploration de données textuelles

### 1-2. Lecture de `txt_file.txt` (UTF-8) et affichage ligne par ligne

In [13]:
with open(find_file("txt_file.txt"), "r", encoding="utf-8") as f:
    for line in f:
        print(line.rstrip("\n"))      # on retire uniquement le saut de ligne final

0.50,0.20,0.84,0.01,c1
0.10,0.10,0.40,0.02,c2
0.15,0.10,0.45,0.03,c2
0.22,0.20,0.90,0.01,c1
0.25,0.20,0.87,0.01,c1
0.10,0.10,0.38,0.02,c2


### 3. Chargement du jeu 20 Newsgroups (sous-ensemble d'entraînement)

Le premier appel peut télécharger les données (connexion internet nécessaire).

In [ ]:
newsgroups = fetch_20newsgroups(subset="train")

### 4. Nombre de documents et d'étiquettes

In [ ]:
print("Nombre de documents :", len(newsgroups.data))
print("Nombre d'étiquettes :", len(newsgroups.target))
print("Forme de target     :", newsgroups.target.shape)

`data` est une liste de documents (un texte par message) et `target` un tableau d'entiers contenant la classe de chaque document. Les deux ont donc la **même longueur** : une étiquette par document.

### 5. Premier document et son étiquette

In [ ]:
print(newsgroups.data[0])
print("-" * 60)
print("Étiquette numérique :", newsgroups.target[0])
print("Classe              :", newsgroups.target_names[newsgroups.target[0]])

### 6. Noms des classes

In [ ]:
print(newsgroups.target_names)
print("Nombre de classes :", len(newsgroups.target_names))

`target_names` est une liste de 20 noms de groupes de discussion. Chaque valeur numérique de `target` est un **indice** dans cette liste : par exemple, `target_names[target[0]]` donne le nom de la classe du premier document.

### 7. (Optionnel) Longueur des documents et mots fréquents

In [ ]:
from collections import Counter
import re

lengths = [len(doc.split()) for doc in newsgroups.data]
print("Longueur moyenne (mots) :", round(np.mean(lengths), 1))
print("Longueur médiane (mots) :", np.median(lengths))
print("Longueur max (mots)     :", max(lengths))

plt.hist(lengths, bins=50, range=(0, 1000))
plt.title("Distribution de la longueur des documents")
plt.xlabel("Nombre de mots")
plt.ylabel("Nombre de documents")
plt.show()

words = re.findall(r"[a-z]+", " ".join(newsgroups.data[:2000]).lower())
Counter(words).most_common(10)

# Exercice 3 — Détection et traitement des valeurs manquantes

### 1. Chargement et inspection de `employees.csv`

In [ ]:
emp = pd.read_csv(find_file("employees.csv"))
emp.info()

In [ ]:
print(emp.dtypes)
emp.head()

### 2. Nombre de valeurs manquantes par colonne

In [ ]:
emp.isna().sum()

### 3. Heatmap des valeurs manquantes (seaborn)

In [ ]:
plt.figure(figsize=(10, 5))
sns.heatmap(emp.isna(), cbar=False, cmap="viridis")
plt.title("Motifs des valeurs manquantes (jaune = manquant)")
plt.xlabel("Colonnes")
plt.ylabel("Observations (index des lignes)")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### 4. Copie et ajout d'une ligne synthétique

La ligne contient les huit champs : `First Name = '-'`, `Gender = 'na'` et les autres champs avec le marqueur de valeur manquante `np.nan`.

In [ ]:
emp2 = emp.copy()

new_row = {col: np.nan for col in emp2.columns}   # tous les champs manquants au départ
new_row[emp2.columns[0]] = "-"                    # first name
new_row[emp2.columns[1]] = "na"                   # gender
print("Ligne ajoutée :", new_row)

emp2 = pd.concat([emp2, pd.DataFrame([new_row])], ignore_index=True)
emp2.tail(3)

### 5. Nouveau comptage des valeurs manquantes

In [ ]:
emp2.isna().sum()

`'-'` et `'na'` sont des **chaînes de caractères** valides : pandas ne considère comme manquantes que certaines valeurs reconnues (`NaN`, `None`, `NaT`, et quelques chaînes lues par `read_csv` comme `""`, `"NA"`, `"NaN"`, `"null"`...). Une chaîne comme `'-'` n'est pas reconnue, et `'na'` en minuscules non plus quand elle est ajoutée à la main dans un DataFrame, car la conversion automatique ne s'applique qu'à la lecture du fichier. Elles restent donc comptées comme des valeurs normales dans `isna()`.

### 6. Conversion des marqueurs en `np.nan`

Règle documentée : pour chaque colonne texte, on retire les espaces autour de la valeur, on la passe en minuscules, et si elle appartient à `{'-', 'na', 'n/a', ''}` elle devient `np.nan`. La règle est donc insensible à la casse.

In [ ]:
markers = {"-", "na", "n/a", ""}

for col in emp2.select_dtypes(include=["object", "string"]).columns:
    mask = emp2[col].astype("string").str.strip().str.lower().isin(markers)
    emp2.loc[mask, col] = np.nan

emp2.isna().sum()

### 7. Imputation avec `SimpleImputer`

- Colonne **identifiant** (prénom) : non imputée, car inventer un prénom n'a aucun sens.
- Colonnes **catégorielles** : `most_frequent`.
- Colonnes **numériques** : `median` (robuste aux valeurs extrêmes).

In [ ]:
id_cols = [emp2.columns[0]]                                   # identifiant : on n'impute pas
num_cols = [c for c in emp2.select_dtypes(include="number").columns
            if emp2[c].nunique() > 2]                  # une colonne à 2 valeurs (0/1) est traitée comme catégorielle
cat_cols = [c for c in emp2.columns if c not in num_cols + id_cols]
print("Identifiant   :", id_cols)
print("Numériques    :", num_cols)
print("Catégorielles :", cat_cols)

emp_imputed = emp2.copy()

emp_imputed[num_cols] = SimpleImputer(strategy="median").fit_transform(emp_imputed[num_cols])
emp_imputed[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(emp_imputed[cat_cols])

### 8. Comparaison avant / après

In [ ]:
pd.DataFrame({
    "Avant imputation": emp2.isna().sum(),
    "Après imputation": emp_imputed.isna().sum(),
})

**Limites de l'imputation :**

- Les valeurs imputées sont des **estimations**, pas des vraies données : la médiane ou le mode ne reflètent pas la situation réelle de chaque ligne.
- L'imputation par une valeur unique **réduit la variance** et peut fausser les corrélations entre variables.
- Si les valeurs ne manquent pas au hasard (par exemple un salaire non renseigné pour une raison précise), l'imputation peut introduire un **biais**.
- Quand il y a beaucoup de valeurs manquantes dans une colonne, il vaut parfois mieux la supprimer ou utiliser une méthode plus fine (KNN, imputation itérative).

**Bonne pratique (éviter la fuite de données).** Quand on prépare des données pour un modèle prédictif, l'imputer doit être ajusté (`fit`) sur le train uniquement, puis appliqué (`transform`) au test :

In [ ]:
train_df, test_df = train_test_split(emp_imputed[num_cols], test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
imputer.fit(train_df)                         # statistiques calculées sur le train seulement
test_imputed = imputer.transform(test_df)     # réutilisées sur le test
print("Médianes apprises sur le train :", dict(zip(num_cols, imputer.statistics_)))

# Exercice 4 — Mise à l'échelle, binarisation et encodage

## Partie A : mise à l'échelle et binarisation

In [ ]:
X_num = np.array([[1, -1,  2],
                  [2,  0,  0],
                  [0,  1, -1]], dtype=float)
cols = ["Feature 1", "Feature 2", "Feature 3"]
pd.DataFrame(X_num, columns=cols)

### 1-2. Application des méthodes et affichage des résultats

In [ ]:
results = {
    "StandardScaler": StandardScaler().fit_transform(X_num),
    "MinMaxScaler": MinMaxScaler().fit_transform(X_num),
    "MaxAbsScaler": MaxAbsScaler().fit_transform(X_num),
    "RobustScaler": RobustScaler().fit_transform(X_num),
    "normalize (L2, par ligne)": normalize(X_num, norm="l2"),
}
for name, res in results.items():
    print(f"=== {name} ===")
    print(pd.DataFrame(res, columns=cols).round(4))
    print()

### 3. Idée principale de chaque méthode

| Méthode | Idée | Agit sur |
|---|---|---|
| `StandardScaler` | Centre et réduit : (x − moyenne) / écart-type. Moyenne 0, écart-type 1. | chaque **variable** (colonne) |
| `MinMaxScaler` | Ramène les valeurs dans [0, 1] : (x − min) / (max − min). | chaque **variable** |
| `MaxAbsScaler` | Divise par la valeur absolue maximale, donc résultats dans [−1, 1], sans centrage (préserve les zéros). | chaque **variable** |
| `RobustScaler` | Soustrait la médiane et divise par l'écart interquartile, donc peu sensible aux valeurs aberrantes. | chaque **variable** |
| `normalize` | Divise chaque ligne par sa norme (L2 par défaut), pour que chaque vecteur ait une norme 1. | chaque **échantillon** (ligne) |

### 4. Binarisation avec un seuil explicite

In [ ]:
for thr in [0.0, 1.0]:
    B = Binarizer(threshold=thr).fit_transform(X_num)
    print(f"Seuil = {thr} (1 si x > seuil, sinon 0)")
    print(pd.DataFrame(B, columns=cols).astype(int))
    print()

La binarisation met à 1 les valeurs **strictement supérieures** au seuil et à 0 les autres. Plus le seuil est élevé, moins il y a de 1 : avec un seuil de 0, seules les valeurs positives deviennent 1, avec un seuil de 1, il ne reste que les valeurs supérieures à 1.

**Discussion : pourquoi la mise à l'échelle est-elle importante ?**

- **k-NN et SVM** reposent sur des **distances** (ou des produits scalaires) : une variable à grande échelle (par exemple un salaire en milliers) domine les autres et fausse les voisinages ou la marge.
- Les **modèles à gradient** (régression logistique, réseaux de neurones) convergent plus vite et plus stablement quand les variables ont des échelles comparables, sinon la surface de perte est très allongée.
- Les **arbres de décision** ne l'exigent pas : ils comparent chaque variable à un seuil, et ce découpage est invariant aux transformations monotones comme un changement d'échelle.

## Partie B : encodage des variables catégorielles

In [ ]:
data = pd.DataFrame({
    "Feature 1": ["male", "female", "female", "male"],
    "Feature 2": ["from US", "from Europe", "from Asia", "from Africa"],
    "Feature 3": ["uses Safari", "uses Safari", "uses Firefox", "uses Chrome"],
    "Target": ["US", "France", "Japan", "Algeria"],
})
data

### 1. `OrdinalEncoder` sur les trois variables d'entrée

In [ ]:
X_cat = data[["Feature 1", "Feature 2", "Feature 3"]]

ord_enc = OrdinalEncoder()
X_encoded = ord_enc.fit_transform(X_cat)

print("categories_ :")
for name, cats in zip(X_cat.columns, ord_enc.categories_):
    print(f"  {name}: {list(cats)}")

pd.DataFrame(X_encoded.astype(int), columns=X_cat.columns)

Les nombres attribués sont des **codes de catégories**, assignés par ordre alphabétique (0, 1, 2...). Ils ne représentent pas un classement : `from US` codé 3 n'est pas « plus grand » que `from Africa` codé 0. Un modèle numérique risque pourtant d'interpréter cet ordre comme une vraie relation.

### 2. Encodage de la cible avec `LabelEncoder`

In [ ]:
y_cat = data["Target"]

label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y_cat)

print("Mapping classe -> label :")
for i, name in enumerate(label_enc.classes_):
    print(f"  {name} -> {i}")
print("Cible encodée :", y_encoded)

### 3. `OneHotEncoder` sur la cible

In [ ]:
ohe = OneHotEncoder(sparse_output=False)
y_onehot = ohe.fit_transform(y_cat.to_frame())

print("Catégories :", ohe.categories_[0])
pd.DataFrame(y_onehot.astype(int), columns=ohe.get_feature_names_out(["Target"]))

**Différence :** l'encodage par entiers (label encoding) représente chaque classe par **un seul nombre** (une colonne), ce qui introduit un ordre artificiel. Le one-hot encoding crée **une colonne binaire par classe** (un seul 1 par ligne), sans aucune notion d'ordre entre les classes.

### 4. Pourquoi préférer le one-hot pour les variables d'entrée nominales ?

Pour une variable nominale (sexe, pays, navigateur), il n'existe pas d'ordre naturel. Des codes ordinaux arbitraires imposent à un modèle (régression, k-NN, SVM, réseau) une **relation d'ordre et des distances qui n'existent pas** (Europe − Africa = 1, Asia − Africa = 2...). Le one-hot encoding donne à chaque catégorie sa propre dimension, toutes équidistantes, donc aucune hiérarchie n'est inventée. Son inconvénient est l'augmentation du nombre de colonnes quand il y a beaucoup de catégories.

# Conclusion

| Type de données | Opérations de prétraitement appropriées |
|---|---|
| **Numériques** | Gestion des valeurs manquantes (médiane), mise à l'échelle (`StandardScaler`, `MinMaxScaler`, `RobustScaler` si valeurs aberrantes), binarisation si besoin d'un seuil. |
| **Catégorielles** | Gestion des valeurs manquantes (valeur la plus fréquente), encodage : one-hot pour les variables nominales, ordinal seulement s'il existe un ordre réel ; `LabelEncoder` pour la cible. |
| **Texte** | Lecture avec le bon encodage (UTF-8), exploration (longueurs, mots fréquents), puis vectorisation avant modélisation. |

Dans tous les cas : travailler sur des **copies**, vérifier les types, fixer `random_state`, et ajuster les transformations sur le **train uniquement** pour éviter la fuite de données.